## LLM MCQ Testing with Mistral LLM

This notebook contains code for testing code inconsistency in Mistral LLM

In [ ]:
import os
import sys

In [ ]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [ ]:
from llm_models.code_llms import Mistral
from mcq_inconsistency.mcq_inconsistency_tester import LLMMCQInconsistencyTester
from mcq_inconsistency.prompt_templates.prompt_template import MCQInconsistencyPromptTemplate
from utility.constants import CodeMMLU, LexicalMutations, SyntacticMutations, LogicalMutations, PromptTypes, ReasoningModels, NonReasoningModels

In [ ]:
## Declaring Prompt Type Constants
ZERO_SHOT = PromptTypes.ZERO_SHOT
ONE_SHOT = PromptTypes.ONE_SHOT
FEW_SHOT = PromptTypes.FEW_SHOT

## Declaring Mutation Constants
FOR2WHILE = SyntacticMutations.FOR2WHILE
FOR2ENUMERATE = SyntacticMutations.FOR2ENUMERATE

RANDOM_MUTATION = LexicalMutations.RANDOM
SEQUENTIAL_MUTATION = LexicalMutations.SEQUENTIAL
LITERAL_FORMAT = LexicalMutations.LITERAL_FORMAT

BOOLEAN_LITERAL = LogicalMutations.BOOLEAN_LITERAL
DEMORGAN = LogicalMutations.DEMORGAN
COMMUTATIVE_REORDER = LogicalMutations.COMMUTATIVE_REORDER
CONSTANT_UNFOLD = LogicalMutations.CONSTANT_UNFOLD
CONSTANT_UNFOLD_ADD = LogicalMutations.CONSTANT_UNFOLD_ADD
CONSTANT_UNFOLD_MULT = LogicalMutations.CONSTANT_UNFOLD_MULT

## Declaring Reasoning Model Name Constants
GPT5 = ReasoningModels.GPT5['name']
GPT4O = ReasoningModels.GPT4O['name']

## Declaring Non-Reasoning Model Name Constants
MISTRAL = NonReasoningModels.MISTRAL_SMALL_LATEST['name']

In [ ]:
reasoning_models = [getattr(ReasoningModels, model) for model in dir(ReasoningModels) if not model.startswith("_")]
non_reasoning_models = [getattr(NonReasoningModels, model) for model in dir(NonReasoningModels) if not model.startswith("_")]
print('Reasoning models supported by this framework are:')
for idx, model in enumerate(reasoning_models):
    print(f"{idx+1}: '{model['name']}'")
print('=' * 50)
print('Non-reasoning models supported by this framework are:')
for idx, model in enumerate(non_reasoning_models):
    print(f"{idx+1}: '{model['name']}'")

In [ ]:
task_set = "CodeMMLU_MCQ_code_completion"
llmtester = LLMMCQInconsistencyTester(task_set)

In [ ]:

# %%script false --no-raise-error
mutations=[FOR2WHILE]
prompt_type = ZERO_SHOT
model_name = GPT4O
task_type = CodeMMLU.Tasks.CODE_COMPLETION
mutation_str = "_".join(mutations) if len(mutations) > 0 else "no_mutation"

results_dir =os.path.join(proj_dir, f'results/mcq_inconsistency/{model_name}')
os.makedirs(results_dir, exist_ok=True)

mutation_str = "_".join(mutations) if len(mutations) > 0 else "no_mutation"
output_file_path=f"{results_dir}/{task_set}_{prompt_type}_{mutation_str}.csv"

pass_count = llmtester.run_mcq_inconsistency_test(
    prompt_helper= MCQInconsistencyPromptTemplate().return_appropriate_prompt(prompt_type=prompt_type),
    num_tests=llmtester.question_database.count_documents({}),
    # num_tests= 1,
    prompt_type= prompt_type,
    mutations=mutations,
    output_file_path=output_file_path,
    task_type =task_type,
    task_set="CodeMMLU",
    model_name=model_name,
)